# VEXA direct GPU endpoint

Serves Qwen from this notebook over the ngrok static domain in the config.
There is no Cloudflare Worker, relay or Durable Object anywhere in the path:
OpenHands talks to ngrok, ngrok talks to this notebook.

Before running:

1. Settings -> Accelerator -> **GPU T4 x2**.
2. Settings -> Internet -> **on**.
3. Add the model dataset as input: `black797/vexa-qwen38-q6k`.
4. Run all. The domain is fixed, so the base URL never changes between starts.

In [ ]:
# VEXA Kaggle GPU runtime (generated by the VEXA control plane).
import json as _vexa_json
CONFIG = _vexa_json.loads("{\"dataset\":\"black797/vexa-qwen38-q6k\",\"modelFile\":\"Qwen3.8-27B-Uncensored-Q6_K.gguf\",\"servedModel\":\"qwen3.8-27b-uncensored-q6_k\",\"ctx\":32768,\"parallel\":1,\"verifyFull\":false,\"loadTimeoutSeconds\":1200,\"direct\":true,\"directApiKey\":\"<MODEL_KEY>\",\"announceUrl\":\"\",\"announceKey\":\"\",\"tunnelCommand\":\"\",\"publicUrl\":\"\",\"ngrokAuthtoken\":\"<NGROK_AUTHTOKEN>\",\"ngrokDomain\":\"<NGROK_DOMAIN>\"}")

"""VEXA Kaggle GPU runtime.

Runs inside a private Kaggle script kernel. The Worker injects CONFIG above this
file, registers the session, loads Qwen with llama.cpp across both T4 GPUs, proves
the model answers, then long-polls the Worker for OpenAI-compatible jobs.
"""
import glob
import hashlib
import http.client
import json
import os
import queue
import re
import shutil
import signal
import socket
import subprocess
import sys
import threading
import time
import traceback
import urllib.error
import urllib.request
import zipfile

try:
    CONFIG  # injected by the Worker
except NameError:
    CONFIG = {}

WORKER = str(CONFIG.get("workerUrl", "")).rstrip("/")
RUNTIME_KEY = CONFIG.get("runtimeKey", "")
# Direct mode serves inference straight from this kernel over its own tunnel. The
# control-plane relay costs a Durable Object request per flush, which is what a free
# Cloudflare plan cannot sustain, so this path removes it entirely.
DIRECT = bool(CONFIG.get("direct"))
PORT = 8080
LLAMA_DIR = "/tmp/vexa-llama"

runtime_token = None
stop_event = threading.Event()
ready_event = threading.Event()
log_queue = queue.Queue()
active_jobs = 0
active_lock = threading.Lock()
server_proc = None
server_log = []
server_state = {"gpuLayers": None, "totalLayers": None, "error": None}
last_heartbeat_ok = time.time()


class ApiError(Exception):
    def __init__(self, status, body):
        super().__init__(f"HTTP {status}: {body[:300]}")
        self.status = status
        self.body = body


def _http(url, payload=None, headers=None, timeout=30, method=None):
    data = json.dumps(payload).encode() if payload is not None else None
    hdrs = {"content-type": "application/json", "user-agent": "vexa-runtime/3.0"}
    hdrs.update(headers or {})
    req = urllib.request.Request(url, data=data, headers=hdrs, method=method or ("POST" if data is not None else "GET"))
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            raw = resp.read()
        return json.loads(raw.decode() or "{}")
    except urllib.error.HTTPError as err:
        body = err.read().decode("utf-8", "replace")
        raise ApiError(err.code, body)
    except (urllib.error.URLError, TimeoutError, socket.timeout) as err:
        raise ApiError(0, f"network error: {err}")


def api(path, payload=None, timeout=30, retries=2, token=None):
    last = None
    for attempt in range(retries + 1):
        try:
            return _http(
                WORKER + path,
                payload,
                {"authorization": f"Bearer {token or runtime_token or RUNTIME_KEY}"},
                timeout,
            )
        except ApiError as err:
            last = err
            if err.status in (400, 401, 403, 404, 409):
                raise
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    raise last


def log(message, level="info"):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    server_log.append(line)
    if len(server_log) > 400:
        del server_log[:200]
    log_queue.put({"ts": int(time.time() * 1000), "level": level, "message": str(message)[:2000]})


def phase(name, detail="", progress=None, extra=None):
    if not runtime_token:
        return
    payload = {"phase": name, "detail": detail}
    if progress is not None:
        payload["progress"] = float(progress)
    if extra:
        payload.update(extra)
    try:
        api("/api/runtime/phase", payload, retries=1)
    except Exception as err:
        log(f"phase update failed: {err}", "warn")


def gpu_stats():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used,utilization.gpu,temperature.gpu,power.draw",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=15,
        ).stdout
    except Exception:
        return []
    gpus = []
    for line in out.strip().splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) < 5:
            continue
        def num(v):
            try:
                return float(v)
            except Exception:
                return None
        gpus.append({
            "index": int(num(parts[0]) or 0),
            "name": parts[1],
            "totalMiB": num(parts[2]),
            "usedMiB": num(parts[3]),
            "utilPct": num(parts[4]),
            "tempC": num(parts[5]) if len(parts) > 5 else None,
            "powerW": num(parts[6]) if len(parts) > 6 else None,
        })
    return gpus


def log_sender():
    while not stop_event.is_set():
        batch = []
        deadline = time.time() + 1.0
        while len(batch) < 40 and time.time() < deadline:
            try:
                batch.append(log_queue.get(timeout=max(0.05, deadline - time.time())))
            except queue.Empty:
                break
        if batch:
            try:
                api("/api/runtime/log", {"lines": batch}, retries=1, timeout=15)
            except Exception:
                pass
        else:
            time.sleep(0.2)


def heartbeat_loop():
    global last_heartbeat_ok
    while not stop_event.is_set():
        try:
            payload = {
                "phase": CONFIG.get("_phase") or ("ready" if ready_event.is_set() else "starting"),
                "gpus": gpu_stats(),
                "activeJobs": active_jobs,
                "progress": CONFIG.get("_progress"),
                "detail": CONFIG.get("_detail", ""),
                "server": dict(server_state),
            }
            res = api("/api/runtime/heartbeat", payload, retries=1, timeout=20)
            last_heartbeat_ok = time.time()
            if res.get("stop"):
                log(f"stop requested: {res.get('reason') or 'control plane'}", "warn")
                stop_event.set()
                break
        except Exception as err:
            if time.time() - last_heartbeat_ok > 180:
                log(f"control plane unreachable for 3 minutes ({err}); exiting to protect quota", "error")
                stop_event.set()
                break
        # Heartbeats are billed control-plane requests: the control plane only fails a
        # session that goes quiet for 150s, so poll fast while loading and slowly when idle.
        stop_event.wait(5 if not ready_event.is_set() else 20)


def set_phase(name, detail="", progress=None, extra=None):
    CONFIG["_phase"] = name
    CONFIG["_detail"] = detail
    CONFIG["_progress"] = progress
    phase(name, detail, progress, extra)


def find_input_file(name):
    slug = str(CONFIG.get("dataset", "")).split("/")[-1]
    direct = os.path.join("/kaggle/input", slug, name)
    if os.path.exists(direct):
        return direct
    for root, _dirs, files in os.walk("/kaggle/input"):
        if name in files:
            return os.path.join(root, name)
    return None


def prepare_llama(dataset_dir):
    os.makedirs(LLAMA_DIR, exist_ok=True)
    copied = 0
    for entry in os.listdir(dataset_dir):
        if entry == "llama-server" or entry.startswith("lib") and ".so" in entry:
            src = os.path.join(dataset_dir, entry)
            if not os.path.isfile(src):
                continue
            dst = os.path.join(LLAMA_DIR, entry)
            if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
                shutil.copy2(src, dst)
                copied += 1
            if entry == "llama-server" or entry.endswith(".so") or ".so." in entry:
                os.chmod(dst, 0o755)
    log(f"llama runtime staged in {LLAMA_DIR} ({copied} files copied)")
    return os.path.join(LLAMA_DIR, "llama-server")


def verify_model(model_path):
    expected_bytes = int(CONFIG.get("modelBytes") or 0)
    expected_sha = str(CONFIG.get("modelSha256") or "")
    size = os.path.getsize(model_path)
    if expected_bytes and size != expected_bytes:
        raise RuntimeError(f"model size mismatch: {size} != {expected_bytes}")
    with open(model_path, "rb") as fh:
        magic = fh.read(4)
    if magic != b"GGUF":
        raise RuntimeError("model file is not a GGUF file")
    digest = None
    if expected_sha and CONFIG.get("verifyFull"):
        h = hashlib.sha256()
        read = 0
        started = time.time()
        with open(model_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(64 * 1024 * 1024), b""):
                h.update(chunk)
                read += len(chunk)
                if expected_bytes:
                    set_phase("loading", "verifying model integrity", min(0.15, 0.15 * read / expected_bytes))
        digest = h.hexdigest()
        if digest != expected_sha:
            raise RuntimeError("model SHA-256 mismatch")
        log(f"full SHA-256 verified in {time.time() - started:.1f}s")
    else:
        with open(model_path, "rb") as fh:
            head = hashlib.sha256(fh.read(32 * 1024 * 1024)).hexdigest()
            fh.seek(max(0, size - 32 * 1024 * 1024))
            tail = hashlib.sha256(fh.read(32 * 1024 * 1024)).hexdigest()
        digest = f"head:{head[:16]} tail:{tail[:16]}"
        log("model quick-verified (size + GGUF magic + head/tail digests)")
    return {"bytes": size, "sha256": digest}


def read_server_output(proc):
    layer_re = re.compile(r"offloaded\s+(\d+)\s*/\s*(\d+)\s+layers to GPU")
    repeat_re = re.compile(r"offloading\s+(\d+)\s+repeating layers")
    err_re = re.compile(r"(error|failed|cuda error|out of memory|invalid argument)", re.I)
    for raw in iter(proc.stdout.readline, b""):
        line = raw.decode("utf-8", "replace").rstrip()
        if not line:
            continue
        if line not in server_log[-3:]:
            log(line, "warn" if err_re.search(line) else "info")
        match = layer_re.search(line)
        if match:
            server_state["gpuLayers"] = int(match.group(1))
            server_state["totalLayers"] = int(match.group(2))
        else:
            partial = repeat_re.search(line)
            if partial:
                server_state["gpuLayers"] = int(partial.group(1))
        if "out of memory" in line.lower() or "cuda error" in line.lower():
            server_state["error"] = line[:500]


def log_ldd(path, env=None):
    try:
        out = subprocess.run(["ldd", path], capture_output=True, text=True, timeout=30, env=env).stdout
        missing = [line.strip().split(" ")[0] for line in out.splitlines() if "not found" in line]
        name = os.path.basename(path)
        if missing:
            log(f"ldd {name}: unresolved {', '.join(missing)}", "warn")
        else:
            log(f"ldd {name}: all dependencies resolved")
    except Exception as err:
        log(f"ldd {os.path.basename(path)} failed: {err}", "warn")


def ensure_cublas(dirs):
    names = ("libcublas.so.12", "libcublasLt.so.12")
    if all(any(os.path.exists(os.path.join(d, n)) for d in dirs) for n in names):
        return dirs
    log("libcublas is missing; installing nvidia-cublas-cu12", "warn")
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-warn-script-location", "nvidia-cublas-cu12"],
            timeout=900, check=False,
        )
    except Exception as err:
        log(f"pip install nvidia-cublas-cu12 failed: {err}", "warn")
    try:
        import site
        for root in site.getsitepackages() + [site.getusersitepackages()]:
            candidate = os.path.join(root, "nvidia", "cublas", "lib")
            if os.path.isdir(candidate) and candidate not in dirs:
                dirs.insert(0, candidate)
    except Exception:
        pass
    return dirs


def server_variants(base, ctx_total):
    """Launch configurations, best first; every one keeps thinking out of content.

    GPU1 runs out of VRAM before GPU0 because the output layer lands there, so the
    speculative-decoding variants hand GPU1 a smaller share of the model.
    """
    wide = max(32768, int(ctx_total))
    safe = min(wide, 65536)
    q8 = ["--cache-type-k", "q8_0", "--cache-type-v", "q8_0"]
    auto = ["--reasoning-format", "auto"]
    spec = ["--spec-type", "draft-mtp", "--spec-draft-n-max", "2"]
    batch = ["-b", "4096", "-ub", "1024"]
    fa = ["-fa", "on"]

    def cfg(ctx, split, *extra):
        return base + ["-c", str(ctx), "--tensor-split", split] + fa + q8 + auto + list(extra) + batch

    return [
        (f"spec@{wide}", cfg(wide, "1,0.8", *spec)),
        (f"spec@{int(wide * 0.75)}", cfg(int(wide * 0.75), "1,0.9", *spec)),
        (f"nofspec@{wide}", cfg(wide, "1,1")),
        (f"nofspec@{safe}", cfg(safe, "1,1")),
        (f"plain@{safe}", base + ["-c", str(safe), "--tensor-split", "1,1"] + auto + batch),
    ]


def start_server(server_path, model_path, skip=0):
    global server_proc
    env = dict(os.environ)
    cuda_dirs = [LLAMA_DIR, "/usr/local/cuda/lib64", "/usr/local/cuda-12/lib64", "/usr/local/cuda-12.2/lib64", "/usr/local/cuda-12.4/lib64"]
    try:
        import site
        for root in site.getsitepackages() + [site.getusersitepackages()]:
            for extra in ("nvidia/cublas/lib", "nvidia/cuda_runtime/lib"):
                candidate = os.path.join(root, extra)
                if os.path.isdir(candidate):
                    cuda_dirs.append(candidate)
    except Exception:
        pass
    cuda_dirs = ensure_cublas(cuda_dirs)
    env["LD_LIBRARY_PATH"] = ":".join(cuda_dirs + [env.get("LD_LIBRARY_PATH", "")]).strip(":")
    env["CUDA_VISIBLE_DEVICES"] = "0,1"
    log_ldd(os.path.join(LLAMA_DIR, "libggml-cuda.so"), env)
    base = [
        server_path,
        "-m", model_path,
        "--host", "127.0.0.1",
        "--port", str(PORT),
        "-ngl", "99",
        "--split-mode", "layer",
        "--main-gpu", "0",
        "-np", str(int(CONFIG.get("parallel") or 2)),
        "-t", "4",
        "--threads-batch", "4",
        "--jinja",
        "--no-warmup",
        "--no-webui",
        "--metrics",
        "--alias", str(CONFIG.get("servedModel") or "vexa"),
    ]
    # A tunnel is world-reachable, so the server itself must demand a key.
    if CONFIG.get("_apiKey"):
        base += ["--api-key", str(CONFIG["_apiKey"])]
    variants = server_variants(base, int(CONFIG.get("ctx") or 32768))
    for index in range(max(0, int(skip)), len(variants)):
        label, args = variants[index]
        log(f"starting llama-server [{label}]: " + " ".join(args[1:]))
        proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env, bufsize=0)
        server_proc = proc
        threading.Thread(target=read_server_output, args=(proc,), daemon=True).start()
        deadline = time.time() + 8
        while time.time() < deadline:
            if proc.poll() is not None:
                break
            if server_state.get("gpuLayers"):
                break
            time.sleep(0.5)
        if proc.poll() is None:
            return proc, index
        log(f"llama-server variant {label} exited during startup (code {proc.returncode})", "warn")
        server_state["error"] = None
        time.sleep(1)
    raise RuntimeError("llama-server failed to start with every supported flag set")


def monitor_loading(proc, model_bytes):
    expected_mib = max(1.0, model_bytes / (1024 * 1024))
    while not ready_event.is_set() and proc.poll() is None:
        gpus = gpu_stats()
        used = sum((g.get("usedMiB") or 0) for g in gpus)
        progress = min(0.95, max(0.05, used / expected_mib))
        set_phase("loading", f"loading weights into VRAM ({used:.0f}/{expected_mib:.0f} MiB)", progress)
        time.sleep(3)


def wait_health(timeout=900):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if server_proc and server_proc.poll() is not None:
            raise RuntimeError("llama-server exited while loading")
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5) as resp:
                if resp.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(2)
    raise RuntimeError("llama-server did not become healthy within the startup timeout")


def server_models():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=10) as resp:
            return json.loads(resp.read().decode())
    except Exception:
        return {}


def _completion(payload):
    headers = {"content-type": "application/json"}
    if CONFIG.get("_apiKey"):
        headers["authorization"] = f"Bearer {CONFIG['_apiKey']}"
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/v1/chat/completions",
        data=json.dumps(payload).encode(),
        headers=headers,
    )
    with urllib.request.urlopen(req, timeout=300) as resp:
        return json.loads(resp.read().decode())


def selftest():
    model = str(CONFIG.get("servedModel") or "vexa")
    base = {"model": model, "temperature": 0, "stream": False}
    attempts = [
        {**base, "messages": [{"role": "user", "content": "Reply with exactly one word: READY"}],
         "max_tokens": 32, "chat_template_kwargs": {"enable_thinking": False}},
        {**base, "messages": [{"role": "user", "content": "Say READY"}],
         "max_tokens": 128, "chat_template_kwargs": {"enable_thinking": False}},
        {**base, "messages": [{"role": "user", "content": "Say READY"}], "max_tokens": 128},
    ]
    last = None
    for payload in attempts:
        started = time.time()
        try:
            result = _completion(payload)
        except Exception as err:
            last = {"error": str(err)[:300]}
            continue
        elapsed_ms = (time.time() - started) * 1000
        message = {}
        try:
            message = result["choices"][0]["message"] or {}
        except Exception:
            pass
        # Thinking models may spend the whole budget on reasoning tokens.
        content = (message.get("content") or message.get("reasoning_content") or "").strip()
        if content:
            # The first completion pays CUDA graph/JIT warm-up, so time a second pass.
            try:
                warm_started = time.time()
                warm = _completion(payload)
                warm_message = (warm.get("choices") or [{}])[0].get("message") or {}
                if (warm_message.get("content") or warm_message.get("reasoning_content") or "").strip():
                    result = warm
                    elapsed_ms = (time.time() - warm_started) * 1000
                    content = (warm_message.get("content") or warm_message.get("reasoning_content") or "").strip()
            except Exception as err:
                log(f"warm-up measurement skipped: {err}", "warn")
            timings = result.get("timings") or {}
            return {
                "ttftMs": round(elapsed_ms, 1),
                "tokPerSec": round(float(timings.get("predicted_per_second") or 0), 2),
                "content": content[:120],
                "usage": result.get("usage") or {},
            }
        last = result
    raise RuntimeError("self-test returned an empty completion: " + json.dumps(last)[:400])


THINK_OPEN = "<" + "think>"
THINK_CLOSE = "</" + "think>"


def split_reasoning(text):
    """Separate a leading think block from the answer. Returns (reasoning, content)."""
    if not text:
        return "", ""
    body = text.lstrip()
    if body.startswith(THINK_OPEN):
        body = body[len(THINK_OPEN):]
    if THINK_CLOSE in body:
        reasoning, _, content = body.partition(THINK_CLOSE)
        return reasoning.strip(), content.lstrip("\n")
    return "", text


def sanitize_message(message):
    """Move thinking the server left inside message.content into reasoning_content."""
    if not isinstance(message, dict):
        return
    content = message.get("content")
    if not isinstance(content, str) or (THINK_OPEN not in content and THINK_CLOSE not in content):
        return
    reasoning, clean = split_reasoning(content)
    message["content"] = clean
    if reasoning:
        message["reasoning_content"] = (message.get("reasoning_content") or "") + reasoning


class StreamSanitizer:
    """Per-job splitter so streamed deltas never expose thinking as content."""

    HOLD = len(THINK_CLOSE) - 1

    def __init__(self):
        self.mode = "lead"
        self.buf = ""
        self.started = False

    def feed(self, delta):
        if not isinstance(delta, dict):
            return delta
        text = delta.get("content")
        if not isinstance(text, str) or not text:
            return delta
        out = {k: v for k, v in delta.items() if k != "content"}
        self.buf += text
        reasoning, content = "", ""
        while True:
            if self.mode == "lead":
                head = self.buf.lstrip()
                if not head:
                    break
                if head.startswith(THINK_OPEN):
                    self.buf = self.buf[len(self.buf) - len(head) + len(THINK_OPEN):]
                    self.mode = "think"
                    continue
                if THINK_OPEN.startswith(head):
                    break
                if THINK_CLOSE in self.buf:
                    self.mode = "think"
                    continue
                self.mode = "content"
                continue
            if self.mode == "think":
                if THINK_CLOSE in self.buf:
                    before, _, self.buf = self.buf.partition(THINK_CLOSE)
                    reasoning += before
                    self.mode = "content"
                    continue
                keep = min(len(self.buf), self.HOLD)
                reasoning += self.buf[:len(self.buf) - keep]
                self.buf = self.buf[len(self.buf) - keep:]
                break
            if not self.started:
                self.buf = self.buf.lstrip("\n")
                if not self.buf:
                    break
                self.started = True
            content += self.buf
            self.buf = ""
            break
        if reasoning:
            out["reasoning_content"] = (delta.get("reasoning_content") or "") + reasoning
        if content:
            out["content"] = content
        return out

    def flush(self):
        text, self.buf = self.buf, ""
        if not text:
            return {}
        if self.mode == "think":
            return {"reasoning_content": text}
        return {"content": text.lstrip("\n")}


def flush(job_id, chunks, seq):
    if not chunks:
        return False, seq
    res = api(f"/api/runtime/jobs/{job_id}/chunk", {"chunks": chunks, "seq": seq}, retries=1, timeout=30)
    return bool(res.get("cancel")), seq + len(chunks)


def stream_job(job_id, body):
    conn = http.client.HTTPConnection("127.0.0.1", PORT, timeout=1800)
    conn.request("POST", "/v1/chat/completions", json.dumps(body),
                 {"content-type": "application/json", "accept": "text/event-stream"})
    resp = conn.getresponse()
    if resp.status != 200:
        raise RuntimeError(f"llama-server HTTP {resp.status}: {resp.read().decode('utf-8', 'replace')[:800]}")
    chunks, seq, last = [], 0, time.time()
    usage, cancelled = None, False
    sanitizer = StreamSanitizer()

    def queue(payload):
        nonlocal seq, cancelled, last
        chunks.append(payload)
        # Each flush is one billed Durable Object request on the control plane, and the
        # free plan allows 100k/day (the client long-polls, so it costs one more per flush).
        # Two flushes per second instead of one per token keeps a day of generation in budget.
        if len(chunks) >= 48 or time.time() - last >= 0.5:
            cancelled, seq = flush(job_id, chunks, seq)
            del chunks[:]
            last = time.time()

    for raw in resp:
        line = raw.decode("utf-8", "replace").strip()
        if not line.startswith("data:"):
            continue
        data = line[5:].strip()
        if data:
            try:
                obj = json.loads(data)
            except Exception:
                obj = None
            if isinstance(obj, dict):
                for choice in obj.get("choices") or []:
                    if isinstance(choice, dict) and isinstance(choice.get("delta"), dict):
                        choice["delta"] = sanitizer.feed(choice["delta"])
                if obj.get("usage"):
                    usage = obj["usage"]
                data = json.dumps(obj, separators=(",", ":"))
        queue(data)
        if cancelled:
            break
    tail = sanitizer.flush()
    if tail and not cancelled:
        queue(json.dumps({
            "id": f"chatcmpl-{job_id}", "object": "chat.completion.chunk",
            "created": int(time.time()), "model": body.get("model"),
            "choices": [{"index": 0, "delta": tail, "finish_reason": None}],
        }, separators=(",", ":")))
    if chunks and not cancelled:
        flush(job_id, chunks, seq)
    conn.close()
    api(f"/api/runtime/jobs/{job_id}/done", {"usage": usage, "cancelled": cancelled}, retries=2)


def blocking_job(job_id, body):
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/v1/chat/completions",
        data=json.dumps(body).encode(),
        headers={"content-type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=1800) as resp:
        result = json.loads(resp.read().decode())
    for choice in result.get("choices") or []:
        if isinstance(choice, dict):
            sanitize_message(choice.get("message"))
    api(f"/api/runtime/jobs/{job_id}/done", {
        "result": result,
        "usage": result.get("usage"),
        "timings": result.get("timings"),
    }, retries=2)


def run_job(job):
    global active_jobs
    with active_lock:
        active_jobs += 1
    job_id = job["id"]
    body = dict(job.get("body") or {})
    stream = bool(job.get("stream"))
    body["model"] = str(CONFIG.get("servedModel") or body.get("model") or "vexa")
    body["stream"] = stream
    if stream:
        body.setdefault("stream_options", {"include_usage": True})
    else:
        body.pop("stream_options", None)
    try:
        if stream:
            stream_job(job_id, body)
        else:
            blocking_job(job_id, body)
    except Exception as err:
        log(f"job {job_id} failed: {err}", "error")
        try:
            api(f"/api/runtime/jobs/{job_id}/error", {"error": str(err)[:800]}, retries=1)
        except Exception:
            pass
    finally:
        with active_lock:
            active_jobs -= 1


def poll_loop(slot):
    while not stop_event.is_set():
        try:
            res = api(f"/api/runtime/poll?slot={slot}&wait=25", None, timeout=40, retries=1)
        except ApiError as err:
            if err.status in (401, 403, 404):
                log(f"control plane rejected the runtime token ({err.status})", "error")
                stop_event.set()
                return
            time.sleep(2)
            continue
        except Exception:
            time.sleep(2)
            continue
        if res.get("stop"):
            log("control plane requested shutdown", "warn")
            stop_event.set()
            return
        job = res.get("job")
        if job:
            run_job(job)
        else:
            time.sleep(0.2)


def shutdown(reason):
    stop_event.set()
    log(f"shutdown: {reason}", "warn")
    deadline = time.time() + 600
    while time.time() < deadline:
        with active_lock:
            if active_jobs == 0:
                break
        time.sleep(1)
    if server_proc and server_proc.poll() is None:
        server_proc.terminate()
        try:
            server_proc.wait(20)
        except Exception:
            server_proc.kill()
    if not runtime_token:
        return
    try:
        api("/api/runtime/stopped", {"reason": reason}, retries=2)
    except Exception as err:
        log(f"failed to report stop: {err}", "warn")


def watchdog():
    while not stop_event.is_set():
        if server_proc and server_proc.poll() is not None:
            log(f"llama-server exited unexpectedly (code {server_proc.returncode})", "error")
            if runtime_token:
                try:
                    api("/api/runtime/error", {"error": "llama-server exited unexpectedly"}, retries=1)
                except Exception:
                    pass
            stop_event.set()
            return
        stop_event.wait(5)


CLOUDFLARED_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
NGROK_ZIP_URL = "https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip"
# Connector output also mentions documentation links; only these hosts are endpoints.
TUNNEL_HOSTS = ("trycloudflare.com", "ngrok-free.app", "ngrok-free.dev", "ngrok.app", "ngrok.io",
                "inrok.in", "loca.lt", "localtunnel.me", "serveo.net", "pinggy.link", "pinggy.io", "bore.pub")
URL_RE = re.compile(r"https://[A-Za-z0-9][A-Za-z0-9.-]+\.[A-Za-z]{2,}")
tunnel_proc = None


def tunnel_url(text, allow_any=False):
    """Pick the endpoint out of connector output, never a docs link."""
    for match in URL_RE.finditer(text):
        url = match.group(0)
        if url.split("//", 1)[1].lower().endswith(TUNNEL_HOSTS):
            return url
    if allow_any:
        match = URL_RE.search(text)
        return match.group(0) if match else None
    return None


def ensure_ngrok():
    """ngrok is not in the Kaggle image.

    The binary ships in a small dataset: fetching it over the network here hangs, and
    that stall happens before anything else can start. Download stays as a fallback.
    """
    found = shutil.which("ngrok")
    if found:
        return found
    local = "/tmp/vexa-ngrok"
    if os.path.exists(local) and os.access(local, os.X_OK):
        return local
    for candidate in sorted(glob.glob("/kaggle/input/*/ngrok")) + sorted(glob.glob("/kaggle/input/*/*/ngrok")):
        if os.path.isfile(candidate):
            shutil.copy2(candidate, local)
            os.chmod(local, 0o755)
            log(f"using the ngrok binary from {candidate}")
            return local
    log("downloading ngrok for the endpoint")
    try:
        archive = "/tmp/vexa-ngrok.zip"
        with urllib.request.urlopen(NGROK_ZIP_URL, timeout=90) as response, open(archive, "wb") as fh:
            shutil.copyfileobj(response, fh)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall("/tmp/vexa-ngrok-bin")
        shutil.copy2("/tmp/vexa-ngrok-bin/ngrok", local)
        os.chmod(local, 0o755)
        return local
    except Exception as err:
        log(f"ngrok download failed: {err}", "error")
        return None


def ensure_cloudflared():
    """Find or download the tunnel client (Kaggle images do not ship one)."""
    local = os.environ.get("CLOUDFLARED_BIN") or "/tmp/vexa-cloudflared"
    found = shutil.which("cloudflared")
    if found:
        return found
    if os.path.exists(local) and os.access(local, os.X_OK):
        return local
    log("downloading cloudflared for the direct endpoint")
    try:
        urllib.request.urlretrieve(CLOUDFLARED_URL, local)
        os.chmod(local, 0o755)
        return local
    except Exception as err:
        log(f"cloudflared download failed: {err}", "error")
        return None


def start_tunnel(port, command=None, timeout=120):
    """Open a public HTTPS tunnel to llama-server and return its URL.

    `command` replaces cloudflared with any tunnel service, e.g. a permanent one: its
    output is scraped for the first https URL, which is how hosted connectors report
    themselves. `{port}` in the command is replaced with the llama-server port.
    """
    global tunnel_proc
    token = str(CONFIG.get("ngrokAuthtoken") or os.environ.get("NGROK_AUTHTOKEN") or "").strip()
    allow_any = False
    if command:
        cmd = ["/bin/sh", "-c", command.replace("{port}", str(port))]
        allow_any = True
    elif token:
        exe = ensure_ngrok()
        if not exe:
            return None
        cmd = [exe, "http", str(port), "--authtoken", token, "--log", "stdout", "--log-format", "logfmt"]
        domain = str(CONFIG.get("ngrokDomain") or os.environ.get("NGROK_DOMAIN") or "").strip()
        if domain:
            cmd.append(f"--url=https://{domain}")
    else:
        exe = ensure_cloudflared()
        if not exe:
            return None
        cmd = [exe, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate", "--loglevel", "info"]
    tunnel_proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, text=True,
    )
    found = threading.Event()
    holder = {}

    def watch():
        for line in tunnel_proc.stdout:
            text = line.rstrip()
            url = tunnel_url(text, allow_any)
            if url and not found.is_set():
                holder["url"] = url
                found.set()
            elif re.search(r"level=error|\bERR\b", text):
                log("tunnel: " + text[:200], "warn")
                # A reserved domain accepts one agent. When another session still holds
                # it, ngrok fails at once; waiting out the timeout only wastes the boot.
                if re.search(r"ERR_NGROK_\d+|session closed|failed to start tunnel", text):
                    holder.setdefault("error", text[:200])
                    found.set()

    threading.Thread(target=watch, daemon=True).start()
    found.wait(timeout)
    if holder.get("error"):
        log("tunnel refused: " + holder["error"], "error")
        return None
    return holder.get("url")


def announce(url):
    """Publish the public URL; one D1 write on the Worker, no Durable Object involved."""
    target = str(CONFIG.get("announceUrl") or "").rstrip("/")
    key = str(CONFIG.get("announceKey") or "")
    if not target or not key:
        return
    payload = {"url": url, "model": CONFIG.get("servedModel"), "kernelRef": CONFIG.get("kernelSlug")}
    for attempt in range(4):
        try:
            _http(target + "/api/direct/announce", payload, {"x-runtime-key": key}, 20)
            log("published the direct endpoint")
            return
        except Exception as err:
            log(f"announce attempt {attempt + 1} failed: {err}", "warn")
            time.sleep(3)


def announce_loop(url):
    while not stop_event.is_set():
        stop_event.wait(300)
        if not stop_event.is_set():
            announce(url)


def boot_model():
    """Locate the model and bring llama-server up. Returns (model_info, proof)."""
    set_phase("allocated", "GPU session allocated; locating the persistent model")
    model_path = find_input_file(CONFIG["modelFile"])
    if not model_path:
        raise RuntimeError(f"{CONFIG['modelFile']} not found under /kaggle/input")
    log(f"model found at {model_path}")
    model_info = verify_model(model_path)
    server_path = prepare_llama(os.path.dirname(model_path))
    set_phase("loading", "starting llama.cpp with CUDA layer split across both T4s", 0.05)
    variant, proc = 0, None
    while True:
        proc, variant = start_server(server_path, model_path, variant)
        threading.Thread(target=monitor_loading, args=(proc, model_info["bytes"]), daemon=True).start()
        try:
            wait_health(int(CONFIG.get("loadTimeoutSeconds") or 1200))
            break
        except Exception as err:
            log(f"launch variant {variant} never became healthy ({err}); trying the next one", "warn")
            try:
                proc.terminate()
                proc.wait(timeout=20)
            except Exception:
                pass
            variant += 1
    set_phase("loading", "llama-server healthy; proving the model answers", 0.97)
    proof = selftest()
    log(f"self-test ok in {proof['ttftMs']}ms ({proof['tokPerSec']} tok/s): {proof['content']!r}")
    ready_event.set()
    return model_info, proof


def report_ready(model_info, proof):
    gpus = gpu_stats()
    vram = sum((g.get("usedMiB") or 0) for g in gpus)
    set_phase("ready", "model loaded and verified; API is live", 1.0, {
        "model": {
            "id": CONFIG.get("servedModel"),
            "ctx": int(CONFIG.get("ctx") or 32768),
            "parallel": int(CONFIG.get("parallel") or 2),
            "gpuLayers": server_state.get("gpuLayers"),
            "totalLayers": server_state.get("totalLayers"),
            "vramMiB": vram,
            "bytes": model_info["bytes"],
            "sha256": model_info["sha256"],
            "models": server_models(),
        },
        "metrics": proof,
        "gpus": gpus,
    })


def direct_main():
    """Serve inference from this kernel over its own public tunnel.

    Nothing here touches the Worker, so the Durable Object quota cannot limit it.
    The tunnel is world-reachable, hence the mandatory bearer key on llama-server.
    """
    key = str(CONFIG.get("directApiKey") or os.environ.get("VEXA_DIRECT_KEY") or "").strip()
    if not key:
        key = "vx-" + hashlib.sha256(os.urandom(32)).hexdigest()[:32]
    CONFIG["_apiKey"] = key
    log("VEXA direct mode: serving from this notebook; the relay is not in the path")
    # Claim the domain before loading 22 GB: a reserved domain accepts one agent, so a
    # leftover session makes this bind fail. Failing here costs seconds, not minutes.
    fixed = str(CONFIG.get("publicUrl") or os.environ.get("VEXA_PUBLIC_URL") or "").strip()
    if fixed:
        url = fixed
        log(f"using the configured permanent endpoint {url}")
    else:
        url = start_tunnel(PORT, os.environ.get("VEXA_TUNNEL_CMD") or CONFIG.get("tunnelCommand"))
    if not url:
        raise RuntimeError(
            "could not open a public tunnel for the direct endpoint; another session may "
            "still hold the reserved domain - stop it and start this one again"
        )
    model_info, proof = boot_model()
    threading.Thread(target=watchdog, daemon=True).start()
    base = url.rstrip("/") + "/v1"
    announce(url)
    threading.Thread(target=announce_loop, args=(url,), daemon=True).start()
    rule = "=" * 64
    log(rule)
    log("VEXA DIRECT ENDPOINT - copy these into your OpenAI / OpenHands profile")
    log(f"  base URL : {base}")
    log(f"  API key  : {key}")
    log(f"  model    : {CONFIG.get('servedModel')}")
    log(f"  check    : curl -s {base}/models -H 'authorization: Bearer {key}'")
    log("  this URL changes every time the notebook starts")
    log(rule)
    while not stop_event.is_set():
        stop_event.wait(1)
    shutdown("notebook stopped")


def main():
    global runtime_token
    if DIRECT:
        return direct_main()
    if not WORKER or not RUNTIME_KEY:
        raise RuntimeError("CONFIG.workerUrl and CONFIG.runtimeKey are required")
    log("VEXA runtime booting")
    reg = _http(WORKER + "/api/runtime/register", {
        "kernelRef": CONFIG.get("kernelSlug"),
        "kernelVersion": CONFIG.get("kernelVersion"),
        "nonce": CONFIG.get("sessionNonce"),
        "gpus": gpu_stats(),
        "host": socket.gethostname(),
    }, {"x-runtime-key": RUNTIME_KEY}, 30)
    runtime_token = reg.get("runtimeToken")
    if not runtime_token:
        raise RuntimeError("registration did not return a runtime token")
    if reg.get("stop"):
        log("control plane rejected this session (stop)", "warn")
        return
    for key in ("ctx", "parallel", "servedModel", "modelBytes", "modelSha256",
                "verifyFull", "dataset", "modelFile"):
        if reg.get(key) is not None:
            CONFIG[key] = reg[key]
    threading.Thread(target=log_sender, daemon=True).start()
    threading.Thread(target=heartbeat_loop, daemon=True).start()
    model_info, proof = boot_model()
    report_ready(model_info, proof)
    threading.Thread(target=watchdog, daemon=True).start()
    for slot in range(int(CONFIG.get("parallel") or 2)):
        threading.Thread(target=poll_loop, args=(slot,), daemon=True).start()
    while not stop_event.is_set():
        stop_event.wait(1)
    shutdown("control plane requested shutdown")


def _signal(_signum, _frame):
    stop_event.set()


if __name__ == "__main__":
    signal.signal(signal.SIGTERM, _signal)
    signal.signal(signal.SIGINT, _signal)
    try:
        main()
    except Exception as exc:
        log(f"fatal: {exc}", "error")
        log(traceback.format_exc(), "error")
        try:
            api("/api/runtime/error", {"error": str(exc)[:800], "trace": traceback.format_exc()[:4000]}, retries=1)
        except Exception:
            pass
        sys.exit(1)
